# конспект новостей с помощью Gemini

кидаем ссылку, парсим из нее новость, сохраняем в файл, далее прогоняем через модель по ключу и получаем конспект и сохраняем его в вфайл

In [ ]:
# Установка зависимостей (выполните эту ячейку один раз)
%pip install requests beautifulsoup4 pandas python-dotenv google-generativeai

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
from dotenv import load_dotenv
import google.generativeai as genai

In [ ]:
# ключ из .env файл
load_dotenv()

url = input("Введите ссылку на новость: ")

In [ ]:
# тут парсинг
print(f"Парсинг данных по ссылке: {url}")
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')
title = soup.find('title').text.strip() if soup.find('title') else "Без заголовка"

# берем текст из разных параграфов
paragraphs = soup.find_all('p')
text_content = "\n".join([p.text.strip() for p in paragraphs if p.text.strip()])

df = pd.DataFrame({'url': [url], 'title': [title], 'content': [text_content]})
df.to_csv('input.csv', index=False, encoding='utf-8')
print("✅ Исходные данные успешно сохранены в input.csv")

In [ ]:
# Настройка Gemini API
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Ключ API не найден. Убедитесь, что он добавлен в файл .env")

genai.configure(api_key=api_key)

# Используем новую бесплатную и быструю модель gemini-2.5-flash
model = genai.GenerativeModel('gemini-2.5-flash')

# Формируем промпт
prompt = f"""Пожалуйста, сделай качественное и краткий конспект следующей новости.\n
Заголовок: {title}\n
Текст: {text_content}\n"""

print("Генерация саммаризации с помощью Gemini...")
response = model.generate_content(prompt)
summary = response.text

print("\n=== РЕЗУЛЬТАТ (САММАРИЗАЦИЯ) ===\n")
print(summary)

In [ ]:
# Сохранение результата в текстовый файл
with open('output_results.txt', 'w', encoding='utf-8') as f:
    f.write(summary)
print("\n✅ Результат успешно сохранен в output_results.txt")